In [ ]:
from processor import load_default_patient
from pathlib import Path
import json

# hard code hr bc we cant calculate it 
heart_rate_bpm = 88
resting_hr_bpm = 60

# get patient deets
patient_file = Path("patients/PID-1234.json")
with open(patient_file, 'r') as f:
    metadata = json.load(f)
subject_metadata = load_default_patient()

print(subject_metadata)


{'subject_id': 'PID-20394', 'name': 'Mr David Chen', 'dob': '1968-04-15', 'device': 'Axivity AX3', 'clinical_context': 'Cardiac rehabilitation post-MI, 6 weeks post-discharge'}


In [5]:
from data_processing import process_cwa_to_summary

# get wearable data
data = process_cwa_to_summary(
    cwa_path="data/tiny-sample.cwa",
    subject_metadata=metadata,
    heart_rate_bpm=72,    # from Holter/ECG if available, else omit
    resting_hr_bpm=62,
)

print(data)

Reading file... Done! (0.74s)
Converting to dataframe... Done! (0.02s)
Quality control... Done! (0.05s)
Lowpass filter... Done! (0.10s)
Gravity calibration... Done! (0.07s)
Nonwear detection... Done! (0.09s)
Resampling... Done! (0.06s)
{'subject_id': 'PID-20394', 'name': 'Mr David Chen', 'dob': '1968-04-15', 'clinical_context': 'Cardiac rehabilitation post-MI, 6 weeks post-discharge', 'device': 'Axivity AX3', 'recording_start': '2023-06-08T12:21:04Z', 'recording_end': '2023-06-08T15:19:33Z', 'mean_daily_steps': 2174, 'total_active_minutes': 8, 'mean_daily_sedentary_minutes': 143, 'days_meeting_150min_activity_target': 0, 'mean_heart_rate_bpm': 72, 'resting_hr_bpm': 62}


In [6]:

import actipy

cwa_path="data/tiny-sample.cwa"
# ------------------------------------------------------------------ #
# 1. Read and process the .cwa file
# ------------------------------------------------------------------ #
data, info = actipy.read_device(
    cwa_path,
    lowpass_hz=20,          # low-pass filter at 20 Hz
    calibrate_gravity=True,  # autocalibration (GGIR-style)
    detect_nonwear=True,     # mark non-wear periods as NaN
    resample_hz=50,         # resample to 50 Hz
)

# data is a pandas DataFrame with columns: x, y, z, temperature, light
# Non-wear periods have NaN in x, y, z

Reading file... Done! (0.62s)
Converting to dataframe... Done! (0.01s)
Quality control... Done! (0.05s)
Lowpass filter... Done! (0.08s)
Gravity calibration... Done! (0.07s)
Nonwear detection... Done! (0.08s)
Resampling... Done! (0.06s)


In [7]:
print(data)

                                x         y         z  temperature      light
time                                                                         
2023-06-08 12:21:04.510  0.593697  0.203108  0.453125        21.50   3.558932
2023-06-08 12:21:04.530 -0.491758  0.325370 -0.633455        21.50   3.558932
2023-06-08 12:21:04.550 -0.469814  0.644069 -0.655645        21.50   3.558932
2023-06-08 12:21:04.570 -0.544060  0.726150 -0.736153        21.50   3.558932
2023-06-08 12:21:04.590 -0.607262  0.449070 -0.678283        21.50   3.558932
...                           ...       ...       ...          ...        ...
2023-06-08 15:19:33.850  0.441156 -0.961747 -0.352812        19.85  41.568882
2023-06-08 15:19:33.870  0.473408 -0.763404 -0.277344        19.85  41.568882
2023-06-08 15:19:33.890  0.414786 -0.761980 -0.243212        19.85  41.568882
2023-06-08 15:19:33.910  0.365099 -0.792435 -0.262619        19.85  41.568882
2023-06-08 15:19:33.930  0.370202 -0.977781 -0.092476        19.

In [8]:
print(info)

{'Filename': 'data/tiny-sample.cwa', 'Filesize(MB)': 4.2, 'Device': 'Axivity', 'DeviceID': 43923, 'ReadErrors': 0, 'SampleRate': 100.0, 'ReadOK': 1, 'StartTime': '2023-06-08 12:21:04', 'EndTime': '2023-06-08 15:19:33', 'NumTicks': 1021800, 'WearTime(days)': 0.1211432638888889, 'DataSpan(days)': 0.12395172453703704, 'NumInterrupts': np.int64(1), 'Covers24hOK': 0, 'LowpassOK': 1, 'LowpassCutoff(Hz)': 20, 'CalibNumSamples': 131, 'CalibErrorBefore(mg)': np.float32(32.778923), 'CalibErrorAfter(mg)': np.float32(32.778923), 'CalibNumIters': 0, 'CalibOK': 0, 'NonwearTime(days)': 0.0, 'NumNonwearEpisodes': 0, 'ResampleRate': 50, 'NumTicksAfterResample': 535472}


In [10]:
import numpy as np
# ------------------------------------------------------------------ #
# 2. Compute magnitude and activity flag (if not already present)
# ------------------------------------------------------------------ #
if "magnitude" not in data.columns:
    data["magnitude"] = np.sqrt(data["x"]**2 + data["y"]**2 + data["z"]**2)

# ENMO (Euclidean Norm Minus One) — standard activity proxy
data["enmo"] = np.maximum(data["magnitude"] - 1, 0)

print(data)

                                x         y         z  temperature      light  \
time                                                                            
2023-06-08 12:21:04.510  0.593697  0.203108  0.453125        21.50   3.558932   
2023-06-08 12:21:04.530 -0.491758  0.325370 -0.633455        21.50   3.558932   
2023-06-08 12:21:04.550 -0.469814  0.644069 -0.655645        21.50   3.558932   
2023-06-08 12:21:04.570 -0.544060  0.726150 -0.736153        21.50   3.558932   
2023-06-08 12:21:04.590 -0.607262  0.449070 -0.678283        21.50   3.558932   
...                           ...       ...       ...          ...        ...   
2023-06-08 15:19:33.850  0.441156 -0.961747 -0.352812        19.85  41.568882   
2023-06-08 15:19:33.870  0.473408 -0.763404 -0.277344        19.85  41.568882   
2023-06-08 15:19:33.890  0.414786 -0.761980 -0.243212        19.85  41.568882   
2023-06-08 15:19:33.910  0.365099 -0.792435 -0.262619        19.85  41.568882   
2023-06-08 15:19:33.930  0.3

In [12]:
from datetime import timezone
# ------------------------------------------------------------------ #
# 3. Recording window
# ------------------------------------------------------------------ #
recording_start = data.index[0].to_pydatetime().replace(tzinfo=timezone.utc)
recording_end   = data.index[-1].to_pydatetime().replace(tzinfo=timezone.utc)

total_duration_days = (recording_end - recording_start).total_seconds() / 86400
# Partial days: use floor for 'days' denominator to avoid inflating per-day averages
full_days = max(int(total_duration_days), 1)

print(recording_start)
print(recording_end)
print(full_days)

2023-06-08 12:21:04.510000+00:00
2023-06-08 15:19:33.930000+00:00
1


In [13]:
# ------------------------------------------------------------------ #
# 4. Resample to 1-minute epochs for activity classification
# ------------------------------------------------------------------ #
# Drop non-wear (NaN rows) before epoching
valid = data.dropna(subset=["x", "y", "z"])

epoch_1min = valid["enmo"].resample("1min").mean()

# Activity thresholds (mg) — standard Hildebrand 2014 cut-points for wrist
# Sedentary : ENMO < 0.030 g  →  < 30 mg
# Light     : 0.030 ≤ ENMO < 0.100 g
# MVPA      : ENMO ≥ 0.100 g  →  ≥ 100 mg
SEDENTARY_THRESHOLD = 0.030  # g (ENMO)
MVPA_THRESHOLD      = 0.100  # g

sedentary_minutes = int((epoch_1min < SEDENTARY_THRESHOLD).sum())
active_minutes    = int((epoch_1min >= MVPA_THRESHOLD).sum())   # MVPA minutes

mean_daily_sedentary = round(sedentary_minutes / full_days)
total_active_minutes = active_minutes  # total across recording, not per-day

print(sedentary_minutes)
print(active_minutes)
print(mean_daily_sedentary)
print(total_active_minutes)

143
8
143
8


In [14]:
import pandas as pd
# ------------------------------------------------------------------ #
# 5. 150-minute weekly target → convert to recording period
# ------------------------------------------------------------------ #
# WHO target: 150 min MVPA per week → ~21.4 min/day
# Flag each calendar day that meets ≥21 min MVPA
daily_active = (
    epoch_1min[epoch_1min >= MVPA_THRESHOLD]
    .resample("1D")
    .count()
    .reindex(pd.date_range(recording_start.date(), recording_end.date(), freq="D"), fill_value=0)
)
DAILY_TARGET_MINUTES = 150 / 7  # ≈ 21.4
days_meeting_target = int((daily_active >= DAILY_TARGET_MINUTES).sum())

print(daily_active)
print(days_meeting_target)

2023-06-08    8
Freq: D, Name: enmo, dtype: int64
0


In [15]:
# ------------------------------------------------------------------ #
# 6. Step estimation
# ------------------------------------------------------------------ #
# actipy doesn't include a step counter natively; use peak detection
# on the band-passed vertical (z) axis. Band-pass 0.5–3 Hz captures
# the 1–2 Hz cadence of normal walking; each peak = one heel-strike.
from scipy.signal import butter, sosfilt, find_peaks

sample_rate = info.get("ResampleRate", 50)  # Hz

sos = butter(4, [0.5, 3.0], btype="bandpass", fs=sample_rate, output="sos")
z_filtered = sosfilt(sos, valid["z"].fillna(0).values)

# Steps can't be closer than 0.3 s apart (~200 steps/min max)
min_distance_samples = int(0.3 * sample_rate)
peaks, _ = find_peaks(z_filtered, height=0.1, distance=min_distance_samples)

total_steps      = len(peaks)
mean_daily_steps = round(total_steps / full_days)

print(min_distance_samples)
print(peaks)
print(total_steps)
print(mean_daily_steps)



15
[    26    145    213 ... 523248 523283 523328]
2174
2174


In [19]:
# ------------------------------------------------------------------ #
# 7. Assemble output
# ------------------------------------------------------------------ #
summary = {
    # --- identity (from caller) ---
    "subject_id":       subject_metadata.get("subject_id"),
    "name":             subject_metadata.get("name"),
    "dob":              subject_metadata.get("dob"),
    "clinical_context": subject_metadata.get("clinical_context"),

    # --- device (from .cwa metadata via actipy info dict) ---
    "device": subject_metadata.get("device") or info.get("DeviceID", "Unknown"),

    # --- recording window ---
    "recording_start": recording_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "recording_end":   recording_end.strftime("%Y-%m-%dT%H:%M:%SZ"),

    # --- activity metrics ---
    "mean_daily_steps":               mean_daily_steps,
    "total_active_minutes":           total_active_minutes,
    "mean_daily_sedentary_minutes":   mean_daily_sedentary,
    "days_meeting_150min_activity_target": days_meeting_target,

    # --- heart rate (external source only — AX3 has no HR sensor) ---
    "mean_heart_rate_bpm": heart_rate_bpm,
    "resting_hr_bpm":      resting_hr_bpm,
}

print(summary)

{'subject_id': 'PID-20394', 'name': 'Mr David Chen', 'dob': '1968-04-15', 'clinical_context': 'Cardiac rehabilitation post-MI, 6 weeks post-discharge', 'device': 'Axivity AX3', 'recording_start': '2023-06-08T12:21:04Z', 'recording_end': '2023-06-08T15:19:33Z', 'mean_daily_steps': 2174, 'total_active_minutes': 8, 'mean_daily_sedentary_minutes': 143, 'days_meeting_150min_activity_target': 0, 'mean_heart_rate_bpm': 88, 'resting_hr_bpm': 60}
